In [2]:
import numpy as np

In [3]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

In [4]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y = np.array([
    [0],
    [1],
    [1],
    [0]
])

print("Input:\n", X)
print("\nOutput:\n", y)

Input:
 [[0 0]
 [0 1]
 [1 0]
 [1 1]]

Output:
 [[0]
 [1]
 [1]
 [0]]


In [5]:
def initialize_parameters():
    np.random.seed(42)

    W1 = np.random.randn(2, 2)
    b1 = np.zeros((1, 2))

    W2 = np.random.randn(2, 1)
    b2 = np.zeros((1, 1))

    return W1, b1, W2, b2

In [6]:
def forward_propagation(X, W1, b1, W2, b2):
    hidden_input = np.dot(X, W1) + b1
    hidden_output = sigmoid(hidden_input)

    final_input = np.dot(hidden_output, W2) + b2
    predicted_output = sigmoid(final_input)

    return hidden_output, predicted_output

In [7]:
def compute_loss(y, predicted_output):
    return np.mean((y - predicted_output) ** 2)

In [8]:
def backpropagation(X, y, hidden_output, predicted_output, W2):
    error = y - predicted_output

    d_output = error * sigmoid_derivative(predicted_output)
    hidden_error = np.dot(d_output, W2.T)
    d_hidden = hidden_error * sigmoid_derivative(hidden_output)

    dW2 = np.dot(hidden_output.T, d_output)
    db2 = np.sum(d_output, axis=0, keepdims=True)

    dW1 = np.dot(X.T, d_hidden)
    db1 = np.sum(d_hidden, axis=0, keepdims=True)

    return dW1, db1, dW2, db2

In [9]:
def train_sgd(X, y, epochs=10000, lr=0.1):
    W1, b1, W2, b2 = initialize_parameters()
    losses = []

    for epoch in range(epochs):
        hidden_output, predicted_output = forward_propagation(X, W1, b1, W2, b2)
        loss = compute_loss(y, predicted_output)
        losses.append(loss)

        dW1, db1, dW2, db2 = backpropagation(X, y, hidden_output, predicted_output, W2)

        # SGD Update
        W1 += lr * dW1
        b1 += lr * db1
        W2 += lr * dW2
        b2 += lr * db2

        if epoch % 2000 == 0:
            print(f"SGD Epoch {epoch}, Loss: {loss:.6f}")

    return W1, b1, W2, b2, predicted_output, losses

In [10]:
W1_sgd, b1_sgd, W2_sgd, b2_sgd, output_sgd, losses_sgd = train_sgd(X, y)

print("\nSGD Final Output:\n", output_sgd)
print("\nSGD Binary Predictions:\n", (output_sgd > 0.5).astype(int))

SGD Epoch 0, Loss: 0.255830
SGD Epoch 2000, Loss: 0.245445
SGD Epoch 4000, Loss: 0.153204
SGD Epoch 6000, Loss: 0.133594
SGD Epoch 8000, Loss: 0.129749

SGD Final Output:
 [[0.05300868]
 [0.49554213]
 [0.95091319]
 [0.50319888]]

SGD Binary Predictions:
 [[0]
 [0]
 [1]
 [1]]


In [11]:
def train_rmsprop(X, y, epochs=10000, lr=0.01, beta=0.9, epsilon=1e-8):
    W1, b1, W2, b2 = initialize_parameters()
    losses = []

    # Cache initialization
    sW1 = np.zeros_like(W1)
    sb1 = np.zeros_like(b1)
    sW2 = np.zeros_like(W2)
    sb2 = np.zeros_like(b2)

    for epoch in range(epochs):
        hidden_output, predicted_output = forward_propagation(X, W1, b1, W2, b2)
        loss = compute_loss(y, predicted_output)
        losses.append(loss)

        dW1, db1, dW2, db2 = backpropagation(X, y, hidden_output, predicted_output, W2)

        # RMSprop cache update
        sW1 = beta * sW1 + (1 - beta) * (dW1 ** 2)
        sb1 = beta * sb1 + (1 - beta) * (db1 ** 2)
        sW2 = beta * sW2 + (1 - beta) * (dW2 ** 2)
        sb2 = beta * sb2 + (1 - beta) * (db2 ** 2)

        # Parameter update
        W1 += lr * dW1 / (np.sqrt(sW1) + epsilon)
        b1 += lr * db1 / (np.sqrt(sb1) + epsilon)
        W2 += lr * dW2 / (np.sqrt(sW2) + epsilon)
        b2 += lr * db2 / (np.sqrt(sb2) + epsilon)

        if epoch % 2000 == 0:
            print(f"RMSprop Epoch {epoch}, Loss: {loss:.6f}")

    return W1, b1, W2, b2, predicted_output, losses

In [12]:
W1_rms, b1_rms, W2_rms, b2_rms, output_rms, losses_rms = train_rmsprop(X, y)

print("\nRMSprop Final Output:\n", output_rms)
print("\nRMSprop Binary Predictions:\n", (output_rms > 0.5).astype(int))

RMSprop Epoch 0, Loss: 0.255830
RMSprop Epoch 2000, Loss: 0.125015
RMSprop Epoch 4000, Loss: 0.125013
RMSprop Epoch 6000, Loss: 0.125012
RMSprop Epoch 8000, Loss: 0.125011

RMSprop Final Output:
 [[0.00256625]
 [0.5037508 ]
 [0.99735479]
 [0.5037508 ]]

RMSprop Binary Predictions:
 [[0]
 [1]
 [1]
 [1]]


In [13]:
def train_adam(X, y, epochs=10000, lr=0.01, beta1=0.9, beta2=0.999, epsilon=1e-8):
    W1, b1, W2, b2 = initialize_parameters()
    losses = []

    # First moment
    mW1 = np.zeros_like(W1)
    mb1 = np.zeros_like(b1)
    mW2 = np.zeros_like(W2)
    mb2 = np.zeros_like(b2)

    # Second moment
    vW1 = np.zeros_like(W1)
    vb1 = np.zeros_like(b1)
    vW2 = np.zeros_like(W2)
    vb2 = np.zeros_like(b2)

    for epoch in range(1, epochs + 1):
        hidden_output, predicted_output = forward_propagation(X, W1, b1, W2, b2)
        loss = compute_loss(y, predicted_output)
        losses.append(loss)

        dW1, db1, dW2, db2 = backpropagation(X, y, hidden_output, predicted_output, W2)

        # Update first moment
        mW1 = beta1 * mW1 + (1 - beta1) * dW1
        mb1 = beta1 * mb1 + (1 - beta1) * db1
        mW2 = beta1 * mW2 + (1 - beta1) * dW2
        mb2 = beta1 * mb2 + (1 - beta1) * db2

        # Update second moment
        vW1 = beta2 * vW1 + (1 - beta2) * (dW1 ** 2)
        vb1 = beta2 * vb1 + (1 - beta2) * (db1 ** 2)
        vW2 = beta2 * vW2 + (1 - beta2) * (dW2 ** 2)
        vb2 = beta2 * vb2 + (1 - beta2) * (db2 ** 2)

        # Bias correction
        mW1_hat = mW1 / (1 - beta1 ** epoch)
        mb1_hat = mb1 / (1 - beta1 ** epoch)
        mW2_hat = mW2 / (1 - beta1 ** epoch)
        mb2_hat = mb2 / (1 - beta1 ** epoch)

        vW1_hat = vW1 / (1 - beta2 ** epoch)
        vb1_hat = vb1 / (1 - beta2 ** epoch)
        vW2_hat = vW2 / (1 - beta2 ** epoch)
        vb2_hat = vb2 / (1 - beta2 ** epoch)

        # Adam Update
        W1 += lr * mW1_hat / (np.sqrt(vW1_hat) + epsilon)
        b1 += lr * mb1_hat / (np.sqrt(vb1_hat) + epsilon)
        W2 += lr * mW2_hat / (np.sqrt(vW2_hat) + epsilon)
        b2 += lr * mb2_hat / (np.sqrt(vb2_hat) + epsilon)

        if epoch % 2000 == 0:
            print(f"Adam Epoch {epoch}, Loss: {loss:.6f}")

    return W1, b1, W2, b2, predicted_output, losses

In [14]:
W1_adam, b1_adam, W2_adam, b2_adam, output_adam, losses_adam = train_adam(X, y)

print("\nAdam Final Output:\n", output_adam)
print("\nAdam Binary Predictions:\n", (output_adam > 0.5).astype(int))

Adam Epoch 2000, Loss: 0.125408
Adam Epoch 4000, Loss: 0.125086
Adam Epoch 6000, Loss: 0.125026
Adam Epoch 8000, Loss: 0.125009
Adam Epoch 10000, Loss: 0.125003

Adam Final Output:
 [[0.00163668]
 [0.49999609]
 [0.99837018]
 [0.50000395]]

Adam Binary Predictions:
 [[0]
 [0]
 [1]
 [1]]


In [15]:
print("Final Loss using SGD     :", losses_sgd[-1])
print("Final Loss using RMSprop :", losses_rms[-1])
print("Final Loss using Adam    :", losses_adam[-1])

Final Loss using SGD     : 0.1282265715241046
Final Loss using RMSprop : 0.12501043020881158
Final Loss using Adam    : 0.12500329931831705
